In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
while not (project_root / "src").is_dir() and project_root.parent != project_root:
    project_root = project_root.parent

if not (project_root / "src").is_dir():
    raise RuntimeError("Could not locate the project root.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

In [8]:
from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

config = Config()

sections = parse_word_document("data/raw/Report.docx")
print(f"Word: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — {len(section['content'])} chars")

sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)
print(f"PDF: {len(sections)} sections, {sum(s['type'] == 'table' for s in sections)} tables")
for section in sections[:5]:
    print(f"  [{section['type']}] {section['title'][:50]} — page {section['page']} — {len(section['content'])} chars")



Word: 61 sections, 5 tables
  [text] Abstract — 1746 chars
  [text] LIST OF ABBREVIATIONS AND SYMBOLS — 57 chars
  [table] LIST OF ABBREVIATIONS AND SYMBOLS — 765 chars
  [text] LIST OF ABBREVIATIONS AND SYMBOLS — 51 chars
  [table] LIST OF ABBREVIATIONS AND SYMBOLS — 218 chars
PDF: 57 sections, 8 tables
  [text] ABSTRACT — page -6 — 1282 chars
  [text] LIST OF ABBREVIATIONS AND SYMBOLS — page 0 — 729 chars
  [text] 1 Introduction and Motivation — page 1 — 3848 chars
  [text] 2 Theoretical Basis and Current Situation — page 3 — 650 chars
  [text] 2.1 Optical Coherence Tomography — page 3 — 2661 chars


In [3]:
import pdfplumber
from collections import Counter

with pdfplumber.open("data/raw/HuMengqing.pdf") as pdf:
    size_counter = Counter()
    size_samples = {}

    for page in pdf.pages[9:]:  # Skip the first 9 introductory pages
        for char in page.chars:
            size = round(char["size"], 1)
            size_counter[size] += 1
            # Record only one sample of text for each font size
            if size not in size_samples:
                # Collect text from the same line as samples 
                size_samples[size] = ""
            if len(size_samples[size]) < 60:
                size_samples[size] += char["text"]

    print(f"{'Size':>6} | {'Count':>7} | Sample Text")
    print("-" * 60)
    for size, count in size_counter.most_common():
        sample = size_samples[size].strip()[:50]
        print(f"{size:>6} | {count:>7} | {sample}")

  Size |   Count | Sample Text
------------------------------------------------------------
  10.6 |  100856 | IX Abbreviations AM Additive Manufacturing OCT Opt
  10.0 |    2699 | Figure 2.1 Schematic of a Generic Fiber-optic OCT 
  11.0 |    1150 | 2.1 Optical Coherence Tomography  2.2 Artificial N
  12.0 |     454 | LIST OF ABBREVIATIONS AND SYMBOLS 1 Introduction a
   7.6 |      89 | 0𝑁𝑁𝑁𝑁−1𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁−1𝑁𝑁−𝑥𝑥−1𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑁𝑛𝑛𝑛𝑛𝑁𝑁𝑛𝑛𝑛𝑛𝑛𝑛𝑁𝑁


In [4]:
from src.core.config import Config

from src.document.word_parser import parse_word_document
sections = parse_word_document("data/raw/Report.docx")
print(f"Total: {len(sections)} sections")
print()
for s in sections:
    title = s["title"][:60]
    chars = len(s["content"])
    print(f"  {chars:>6} chars | [{s['type']:>5}] {title}")

Total: 61 sections

    1746 chars | [ text] Abstract
      57 chars | [ text] LIST OF ABBREVIATIONS AND SYMBOLS
     765 chars | [table] LIST OF ABBREVIATIONS AND SYMBOLS
      51 chars | [ text] LIST OF ABBREVIATIONS AND SYMBOLS
     218 chars | [table] LIST OF ABBREVIATIONS AND SYMBOLS
    3714 chars | [ text] 1 Introduction
    2395 chars | [ text] 2.1 Additive Manufacturing
    2735 chars | [ text] 2.2 Optical Coherence Tomography
    1104 chars | [ text] 2.3 Artificial Neural Networks
     716 chars | [ text] 2.3.1 Neuron
     887 chars | [ text] 2.3.2 Activation Function
     538 chars | [ text] 2.3.3 Weights and Biases
     649 chars | [ text] 2.3.4 Layers
     739 chars | [ text] 2.3.5 Loss Function
     813 chars | [ text] 2.3.6 Optimizer
     629 chars | [ text] 2.3.7 Gradient Descent
    1128 chars | [ text] 2.3.8 Backpropagation
    1561 chars | [ text] 2.4 Convolutional Neural Networks
    1126 chars | [ text] 2.4.1 Convolutional Layer
    1394 chars | [ text] 2.4.2 Pooli

In [5]:
from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document

config = Config()
sections = parse_pdf_document("data/raw/HuMengqing.pdf", config)

print(f"Total: {len(sections)} sections")
print()

for section in sections:
    title = section["title"][:60]
    character_count = len(section["content"])
    page_number = section.get("page", "?")

    print(
        f"  page {page_number:>3} | {character_count:>6} chars | "
        f"[{section['type']}] {title}"
    )

Total: 57 sections

  page  -6 |   1282 chars | [text] ABSTRACT
  page   0 |    729 chars | [text] LIST OF ABBREVIATIONS AND SYMBOLS
  page   1 |   3848 chars | [text] 1 Introduction and Motivation
  page   3 |    650 chars | [text] 2 Theoretical Basis and Current Situation
  page   3 |   2661 chars | [text] 2.1 Optical Coherence Tomography
  page   4 |   6525 chars | [text] 2.2.1 Feed-forward Neural Networks
  page   8 |   3480 chars | [text] 2.2.2 Convolutional Neural Networks
  page  10 |   1337 chars | [text] 2.2.3.1 LeNet5
  page  11 |   1454 chars | [text] 2.2.3.2 AlexNet
  page  12 |   1112 chars | [text] 2.2.3.3 VGG
  page  12 |   4378 chars | [text] 2.2.3.4 ResNet
  page  16 |   2415 chars | [text] 2.2.3.5 EfficientNet
  page  17 |    381 chars | [text] 2.2.4 Performance Evaluation
  page  17 |    556 chars | [text] 2.2.4.1 Loss and Accuracy of Training and Validation
  page  17 |    704 chars | [text] 2.2.4.2 Confusion Matrix
  page  18 |   1573 chars | [text] 2.2.4.3 Precisi

In [6]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/v2/sections")
output_directory.mkdir(parents=True, exist_ok=True)

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, Config()
    ),
}

for document_path, parser in documents.items():
    sections = parser(document_path)
    output_path = output_directory / f"{document_path.stem}.json"
    output_path.write_text(
        json.dumps(sections, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(sections)} sections to {output_path}")


Saved 61 sections to output/v2/sections/Report.json
Saved 57 sections to output/v2/sections/HuMengqing.json


In [7]:
import json
from pathlib import Path

from src.core.config import Config
from src.document.chunker import chunk_sections
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document

output_directory = Path("output/v2/chunks")
output_directory.mkdir(parents=True, exist_ok=True)
config = Config()

documents = {
    Path("data/raw/Report.docx"): parse_word_document,
    Path("data/raw/HuMengqing.pdf"): lambda document_path: parse_pdf_document(
        document_path, config
    ),
}
all_chunks = []

for document_path, parser in documents.items():
    sections = parser(document_path)
    chunks = chunk_sections(sections, config)
    all_chunks.extend(chunks)
    output_path = output_directory / f"{document_path.stem}_chunks.json"
    output_path.write_text(
        json.dumps(chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"Saved {len(chunks)} chunks to {output_path}")

all_chunks_path = output_directory / "all_chunks.json"
all_chunks_path.write_text(
    json.dumps(all_chunks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Saved {len(all_chunks)} chunks to {all_chunks_path}")


Saved 142 chunks to output/v2/chunks/Report_chunks.json
Saved 121 chunks to output/v2/chunks/HuMengqing_chunks.json
Saved 263 chunks to output/v2/chunks/all_chunks.json
